# Football Player Market Value Prediction & Scouting Dashboard
## Notebook 2 — Data Cleaning & Merging

**Author:** Ashish Abraham
**Date:** July 2026

The primary analysis uses the 2024-25 season only, with the full 267-column feature set, stratified k-fold CV across four position-specific models (FWD/MID/DEF/GK) and four algorithms (Ridge, RF, XGBoost, LightGBM), explained with SHAP. The feature ablation reuses the same 2024-25 data and folds but restricts to the reduced feature set also present in 2025-26 (~20 columns), to see how much predictive power the advanced metrics are actually adding. 2025-26 itself only comes in as an optional external validation check for the reduced-feature model, since it's missing xG and most of the passing/possession/defense detail. The 6-season scraped data stays descriptive — trend lines and distributions, not modelling.

This notebook produces the merged 2024-25 dataset with valuations joined in, the four position-split dataframes, and the reduced-feature ablation dataframe, all saved to `data/processed/`.

## Step 1 — Load Datasets

Loads `players_data-2024_2025.csv` (primary) and `players_data-2025_2026.csv` (needed only to identify the reduced/ablation feature set — not used as a temporal test set for the primary model). Checks shape, dtypes, and header integrity before any transformation.

In [51]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
from collections import Counter

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

RAW_KAGGLE = Path("../data/raw/fbref_kaggle/")
RAW_TMARKT = Path("../data/raw/transfermarkt/")

df_2425 = pd.read_csv(RAW_KAGGLE / "players_data-2024_2025.csv")
df_2526 = pd.read_csv(RAW_KAGGLE / "players_data-2025_2026.csv")

print("=== players_data-2024_2025.csv (PRIMARY) ===")
print(f"shape: {df_2425.shape}")
print(f"dtypes summary:\n{df_2425.dtypes.value_counts()}")

print("\n=== players_data-2025_2026.csv (reduced-feature reference only) ===")
print(f"shape: {df_2526.shape}")
print(f"dtypes summary:\n{df_2526.dtypes.value_counts()}")

=== players_data-2024_2025.csv (PRIMARY) ===
shape: (2854, 267)
dtypes summary:
int64      121
float64    111
str         35
Name: count, dtype: int64

=== players_data-2025_2026.csv (reduced-feature reference only) ===
shape: (2839, 102)
dtypes summary:
float64    45
int64      40
str        17
Name: count, dtype: int64


In [52]:
print("2024-25 first row:")
print(df_2425.iloc[0][['Player', 'Squad', 'Comp']])
print("\n2025-26 first row:")
print(df_2526.iloc[0][['Player', 'Squad', 'Comp']])

print(f"\nduplicate Player rows in 2024-25: {df_2425['Player'].duplicated().sum()}")
print(f"duplicate Player rows in 2025-26: {df_2526['Player'].duplicated().sum()}")
print(f"\ntransfermarkt files found: {list(RAW_TMARKT.glob('*.csv'))[:5]}")

2024-25 first row:
Player            Max Aarons
Squad            Bournemouth
Comp      eng Premier League
Name: 0, dtype: object

2025-26 first row:
Player      Brenden Aaronson
Squad           Leeds United
Comp      eng Premier League
Name: 0, dtype: object

duplicate Player rows in 2024-25: 152
duplicate Player rows in 2025-26: 152

transfermarkt files found: [PosixPath('../data/raw/transfermarkt/game_lineups.csv'), PosixPath('../data/raw/transfermarkt/competitions.csv'), PosixPath('../data/raw/transfermarkt/appearances.csv'), PosixPath('../data/raw/transfermarkt/countries.csv'), PosixPath('../data/raw/transfermarkt/player_valuations.csv')]


## Step 2 — Resolve Internal Suffix-Duplicate Columns (2024-25)

The 2024-25 file merges 10 FBref stat tables per player, so identity and stat columns repeat under suffixes (`_stats_shooting`, `_stats_defense`, etc.) whenever they collide with a name already claimed by an earlier table. Classify each suffix group as a true duplicate (safe to collapse), a partial match (rename the plain column to its real source, keep the rest), or ambiguous (needs a closer look) before touching anything.

In [53]:
suffix_pattern = re.compile(r'^(.*)_stats_([a-z_]+)$')

def find_suffix_groups(df):
    groups = {}
    for col in df.columns:
        m = suffix_pattern.match(col)
        if m:
            groups.setdefault(m.group(1), []).append(col)
    return groups

groups_2425 = find_suffix_groups(df_2425)
THRESH = 0.999

def classify_suffix_groups(df, groups):
    classification = {}
    for base, suffix_cols in groups.items():
        if base == 'Rk':
            classification[base] = {'type': 'rk', 'agreements': {}, 'high_agree': []}
            continue
        if base not in df.columns:
            classification[base] = {'type': 'no_plain', 'agreements': {}, 'high_agree': []}
            continue
        agreements = {}
        for sc in suffix_cols:
            mask = df[base].notna() & df[sc].notna()
            agreements[sc] = np.nan if mask.sum() == 0 else (df.loc[mask, base] == df.loc[mask, sc]).mean()
        high_agree = [sc for sc, pct in agreements.items() if pd.notna(pct) and pct >= THRESH]
        ctype = 'true_dup' if len(high_agree) == len(suffix_cols) else ('partial_id' if high_agree else 'ambiguous')
        classification[base] = {'type': ctype, 'agreements': agreements, 'high_agree': high_agree}
    return classification

class_2425 = classify_suffix_groups(df_2425, groups_2425)

print(Counter(info['type'] for info in class_2425.values()))
print("\ntrue_dup :", sorted(b for b, i in class_2425.items() if i['type'] == 'true_dup'))
print("\npartial_id:", {b: i['high_agree'] for b, i in class_2425.items() if i['type'] == 'partial_id'})
print("\nambiguous:", sorted(b for b, i in class_2425.items() if i['type'] == 'ambiguous'))

Counter({'true_dup': 22, 'ambiguous': 16, 'partial_id': 6, 'rk': 1})

true_dup : ['90s', 'Age', 'Ast', 'Born', 'Comp', 'CrdR', 'CrdY', 'Crs', 'GA', 'Gls', 'Int', 'Nation', 'PK', 'PKA', 'Pos', 'PrgC', 'PrgP', 'PrgR', 'TklW', 'npxG', 'xAG', 'xG']

partial_id: {'PKatt': ['PKatt_stats_shooting'], 'Att': ['Att_stats_passing_types'], 'Cmp': ['Cmp_stats_passing_types'], 'MP': ['MP_stats_playing_time'], 'Min': ['Min_stats_playing_time'], 'Starts': ['Starts_stats_playing_time']}

ambiguous: ['1/3', 'Att 3rd', 'Blocks', 'CK', 'Cmp%', 'Def 3rd', 'FK', 'Fld', 'Live', 'Lost', 'Mid 3rd', 'OG', 'Off', 'PrgDist', 'Sh', 'TotDist']


## Step 3 — Resolve Ambiguous Columns via FBref Source-Table Schema

Cross-reference each ambiguous base name against known FBref stat-table schemas. The plain column is whichever schema-plausible table isn't already claimed by a suffix variant (a suffix only exists because it collided with an earlier-merged table).

In [54]:
fbref_schema = {
    'shooting':      ['Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Dist', 'FK', 'PK', 'PKatt'],
    'passing':       ['Cmp', 'Att', 'Cmp%', 'TotDist', 'PrgDist', 'KP', '1/3', 'PPA', 'CrsPA', 'PrgP'],
    'passing_types': ['Att', 'Live', 'Dead', 'FK', 'TB', 'Sw', 'Crs', 'TI', 'CK', 'In', 'Out', 'Str', 'Cmp', 'Off', 'Blocks'],
    'defense':       ['Tkl', 'TklW', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Att', 'Tkl%', 'Lost', 'Blocks', 'Sh', 'Pass', 'Int', 'Clr', 'Err'],
    'possession':    ['Touches', 'Def Pen', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Att Pen', 'Live', 'Att', 'Succ', 'Succ%',
                       'Tkld', 'Tkld%', 'Carries', 'TotDist', 'PrgDist', '1/3', 'CPA', 'Mis', 'Dis', 'Rec'],
    'gca':           ['SCA', 'SCA90', 'GCA', 'GCA90', 'PassLive', 'PassDead', 'TO', 'Sh', 'Fld', 'Def'],
    'misc':          ['CrdY', 'CrdR', '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'PKwon', 'PKcon', 'OG', 'Recov', 'Lost'],
    'keeper_adv':    ['GA', 'PKA', 'FK', 'CK', 'OG', 'PSxG', 'Cmp', 'Att', 'Cmp%', 'Launch%', 'AvgLen'],
}

ambiguous_cols = sorted(b for b, i in class_2425.items() if i['type'] == 'ambiguous')
existing_suffixes = {base: groups_2425[base] for base in ambiguous_cols}

for col in ambiguous_cols:
    schema_matches = [tbl for tbl, cols in fbref_schema.items() if col in cols]
    existing = [s.replace(f"{col}_stats_", "") for s in existing_suffixes[col]]
    remaining = [tbl for tbl in schema_matches if tbl not in existing]
    status = remaining[0] if len(remaining) == 1 else (remaining if remaining else 'STILL UNRESOLVED')
    print(f"{col:10s} | schema: {schema_matches} | already suffixed: {existing} | plain col resolves to: {status}")

1/3        | schema: ['passing', 'possession'] | already suffixed: ['possession'] | plain col resolves to: passing
Att 3rd    | schema: ['defense', 'possession'] | already suffixed: ['possession'] | plain col resolves to: defense
Blocks     | schema: ['passing_types', 'defense'] | already suffixed: ['defense'] | plain col resolves to: passing_types
CK         | schema: ['passing_types', 'keeper_adv'] | already suffixed: ['keeper_adv'] | plain col resolves to: passing_types
Cmp%       | schema: ['passing', 'keeper_adv'] | already suffixed: ['keeper_adv'] | plain col resolves to: passing
Def 3rd    | schema: ['defense', 'possession'] | already suffixed: ['possession'] | plain col resolves to: defense
FK         | schema: ['shooting', 'passing_types', 'keeper_adv'] | already suffixed: ['passing_types', 'keeper_adv'] | plain col resolves to: shooting
Fld        | schema: ['gca', 'misc'] | already suffixed: ['misc'] | plain col resolves to: gca
Live       | schema: ['passing_types', 'posses

## Step 4 — Empirical Sanity Check on Resolved Ambiguous Columns

Spot-check a handful of resolved columns against what their claimed source table should actually look like — row coverage and value range — before trusting the schema-based resolution.

In [55]:
gk_mask = df_2425['Pos'].str.contains('GK', na=False)

print("=== Cmp% (resolved: passing) ===")
print(f"non-null overall: {df_2425['Cmp%'].notna().mean():.1%} | GK only: {df_2425.loc[gk_mask,'Cmp%'].notna().mean():.1%} | outfield only: {df_2425.loc[~gk_mask,'Cmp%'].notna().mean():.1%}")
print(f"range: {df_2425['Cmp%'].min()} to {df_2425['Cmp%'].max()}")

print("\n=== Sh (resolved: shooting) ===")
print(df_2425.loc[~gk_mask, 'Sh'].describe())

print("\n=== FK (resolved: shooting) ===")
print(df_2425.loc[~gk_mask, 'FK'].describe())

print("\n=== OG (resolved: misc) ===")
print(f"non-null among GKs: {df_2425.loc[gk_mask,'OG'].notna().mean():.1%}")
print(df_2425['OG'].value_counts().head())

=== Cmp% (resolved: passing) ===
non-null overall: 99.3% | GK only: 100.0% | outfield only: 99.3%
range: 0.0 to 100.0

=== Sh (resolved: shooting) ===
count    2642.000000
mean       16.331945
std        19.267403
min         0.000000
25%         3.000000
50%        10.000000
75%        23.000000
max       152.000000
Name: Sh, dtype: float64

=== FK (resolved: shooting) ===
count    2642.000000
mean        0.474640
std         1.505864
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        20.000000
Name: FK, dtype: float64

=== OG (resolved: misc) ===
non-null among GKs: 100.0%
OG
0    2721
1     121
2      10
3       2
Name: count, dtype: int64


## Step 5 — Apply All Column Resolutions

Applies every fix in one pass: true duplicates collapse to the plain column; partial matches get the plain column renamed to its confirmed source *and* the one suffix variant that agreed with it dropped (it's now a redundant copy); ambiguous columns get renamed to their schema-confirmed source; `Rk` is dropped outright.

In [56]:
df_2425_clean = df_2425.copy()

# 1. true_dup — drop suffixed copies, keep the plain column
true_dup_bases = [b for b, i in class_2425.items() if i['type'] == 'true_dup']
cols_to_drop = [c for base in true_dup_bases for c in groups_2425[base]]
df_2425_clean = df_2425_clean.drop(columns=cols_to_drop)
print(f"true_dup: dropped {len(cols_to_drop)} suffixed columns, kept {len(true_dup_bases)} plain columns")

# 2. Rk — drop entirely, it's a row-rank artifact
rk_cols = groups_2425['Rk'] + (['Rk'] if 'Rk' in df_2425_clean.columns else [])
df_2425_clean = df_2425_clean.drop(columns=[c for c in rk_cols if c in df_2425_clean.columns])
print(f"Rk: dropped {len(rk_cols)} columns")

# 3. partial_id — rename plain column to its confirmed source, AND drop the
#    specific suffix variant that agreed with it (now a redundant duplicate)
partial_id_rename = {
    'PKatt': 'PKatt_shooting', 'Att': 'Att_passing_types', 'Cmp': 'Cmp_passing_types',
    'MP': 'MP_playing_time', 'Min': 'Min_playing_time', 'Starts': 'Starts_playing_time',
}
redundant_suffix_cols = [c for base in partial_id_rename for c in class_2425[base]['high_agree']]
df_2425_clean = df_2425_clean.drop(columns=redundant_suffix_cols)
df_2425_clean = df_2425_clean.rename(columns=partial_id_rename)
print(f"partial_id: renamed {len(partial_id_rename)} columns, dropped {len(redundant_suffix_cols)} now-redundant duplicates")

# 4. ambiguous — rename plain column; nothing redundant to drop here
ambiguous_resolution = {
    '1/3': 'passing', 'Att 3rd': 'defense', 'Blocks': 'passing_types', 'CK': 'passing_types',
    'Cmp%': 'passing', 'Def 3rd': 'defense', 'FK': 'shooting', 'Fld': 'gca',
    'Live': 'passing_types', 'Lost': 'defense', 'Mid 3rd': 'defense', 'OG': 'misc',
    'Off': 'passing_types', 'PrgDist': 'passing', 'Sh': 'shooting', 'TotDist': 'passing',
}
df_2425_clean = df_2425_clean.rename(columns={b: f"{b}_{t}" for b, t in ambiguous_resolution.items()})
print(f"ambiguous: renamed {len(ambiguous_resolution)} columns to reflect schema-confirmed source")

print(f"\nfinal shape: {df_2425_clean.shape}")
print(f"duplicate column names? {df_2425_clean.columns.duplicated().any()}")

true_dup: dropped 76 suffixed columns, kept 22 plain columns
Rk: dropped 11 columns
partial_id: renamed 6 columns, dropped 6 now-redundant duplicates
ambiguous: renamed 16 columns to reflect schema-confirmed source

final shape: (2854, 174)
duplicate column names? False


## Step 6 — Standardise Remaining Suffix Naming

Strips `stats_` from every remaining suffixed column so naming is consistent everywhere.

In [57]:
rename_remaining = {c: c.replace('_stats_', '_') for c in df_2425_clean.columns if '_stats_' in c}
df_2425_clean = df_2425_clean.rename(columns=rename_remaining)

print(f"standardised {len(rename_remaining)} column names")
print(f"columns still using old format: {sum('_stats_' in c for c in df_2425_clean.columns)}")
print(f"duplicate column names? {df_2425_clean.columns.duplicated().any()}")
print(f"final shape: {df_2425_clean.shape}")

standardised 26 column names
columns still using old format: 0
duplicate column names? False
final shape: (2854, 174)


## Step 7 — Minimum Minutes Filter

Drop players with fewer than 3 × 90s (270 minutes) in 2024-25 — too little playing time for meaningful per-90 stats.

In [58]:
below_threshold = df_2425_clean['90s'] < 3.0
print(f"before filter: {df_2425_clean.shape[0]} | below threshold: {below_threshold.sum()} ({below_threshold.mean():.1%})")

df_2425_filtered = df_2425_clean[df_2425_clean['90s'] >= 3.0].copy()
print(f"after filter: {df_2425_filtered.shape[0]}")

print(f"\nposition distribution before:\n{df_2425_clean['Pos'].value_counts(normalize=True).head(10)}")
print(f"\nposition distribution after:\n{df_2425_filtered['Pos'].value_counts(normalize=True).head(10)}")

before filter: 2854 | below threshold: 640 (22.4%)
after filter: 2214

position distribution before:
Pos
DF       0.300981
MF       0.206377
FW       0.129993
FW,MF    0.113875
MF,FW    0.080589
GK       0.074282
DF,MF    0.038542
MF,DF    0.028381
DF,FW    0.018570
FW,DF    0.008409
Name: proportion, dtype: float64

position distribution after:
Pos
DF       0.317073
MF       0.198284
FW       0.120596
FW,MF    0.115628
MF,FW    0.078139
GK       0.076784
DF,MF    0.039747
MF,DF    0.029359
DF,FW    0.019422
FW,DF    0.004968
Name: proportion, dtype: float64


## Step 8 — Position Cleaning

Maps FBref's combined `Pos` values (e.g. `MF,FW`) to a single primary position group — FWD, MID, DEF, GK — using the first-listed position.

In [59]:
def map_primary_position(pos):
    if pd.isna(pos):
        return np.nan
    first = pos.split(',')[0].strip()
    return {'GK': 'GK', 'DF': 'DEF', 'MF': 'MID', 'FW': 'FWD'}.get(first, np.nan)

df_2425_filtered['position_group'] = df_2425_filtered['Pos'].apply(map_primary_position)
print(df_2425_filtered['position_group'].value_counts())
print(f"\nunmapped rows: {df_2425_filtered['position_group'].isna().sum()}")

position_group
DEF    833
MID    677
FWD    534
GK     170
Name: count, dtype: int64

unmapped rows: 0


## Step 9 — Build FBref ↔ Transfermarkt Player Mapping

No pre-built mapping file exists, so it's built from scratch against `players.csv` — exact match first, then fuzzy matching (`thefuzz`, score ≥ 85) on the remainder.

In [60]:
from thefuzz import fuzz, process

df_players_tm = pd.read_csv(RAW_TMARKT / "players.csv")
print(f"transfermarkt players.csv shape: {df_players_tm.shape}")

tm_name_counts = df_players_tm['name'].value_counts()
tm_duplicate_names = tm_name_counts[tm_name_counts > 1]
print(f"duplicate names in transfermarkt data: {len(tm_duplicate_names)}")

tm_lookup = df_players_tm.drop_duplicates(subset='name', keep='first').set_index('name')['player_id'].to_dict()

df_2425_filtered['tm_player_id'] = df_2425_filtered['Player'].map(tm_lookup)
df_2425_filtered['match_method'] = np.where(df_2425_filtered['tm_player_id'].notna(), 'exact', None)

exact_match_count = df_2425_filtered['tm_player_id'].notna().sum()
print(f"exact matches: {exact_match_count} / {len(df_2425_filtered)} ({exact_match_count/len(df_2425_filtered):.1%})")

transfermarkt players.csv shape: (48380, 26)
duplicate names in transfermarkt data: 894
exact matches: 1921 / 2214 (86.8%)


## Step 10 — Check for Accent Mismatches Before Fuzzy Matching

Some unmatched names may just be missing accents versus Transfermarkt's spelling. Separately, check how many "exact" matches landed on a duplicated Transfermarkt name, since taking the first match blindly risks the wrong player.

In [61]:
def strip_accents(s):
    if pd.isna(s):
        return s
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

unmatched = df_2425_filtered[df_2425_filtered['tm_player_id'].isna()].copy()
print(f"unmatched after exact pass: {len(unmatched)}")

tm_lookup_stripped = {strip_accents(name): pid for name, pid in tm_lookup.items()}
unmatched['accent_stripped_match'] = unmatched['Player'].apply(strip_accents).map(tm_lookup_stripped)
print(f"recoverable via accent-stripping: {unmatched['accent_stripped_match'].notna().sum()}")

matched_names = df_2425_filtered.loc[df_2425_filtered['tm_player_id'].notna(), 'Player']
print(f"exact matches exposed to a duplicated TM name: {matched_names.isin(tm_duplicate_names.index).sum()}")

unmatched after exact pass: 293
recoverable via accent-stripping: 117
exact matches exposed to a duplicated TM name: 74


## Step 11 — Apply Accent Matches, Check Duplicate Exposure (First Pass)

Applies the accent-recovered matches, then checks how many matches are exposed to a duplicated Transfermarkt name using the duplicate list built earlier — on *raw* (accented) names.

In [62]:
newly_matched = unmatched[unmatched['accent_stripped_match'].notna()].copy()
df_2425_filtered.loc[newly_matched.index, 'tm_player_id'] = newly_matched['accent_stripped_match']
df_2425_filtered.loc[newly_matched.index, 'match_method'] = 'exact_accent_stripped'
print(f"matches after accent-stripping: {df_2425_filtered['tm_player_id'].notna().sum()} / {len(df_2425_filtered)}")

matched_mask = df_2425_filtered['tm_player_id'].notna()
exposed_mask_naive = matched_mask & df_2425_filtered['Player'].apply(strip_accents).isin(tm_duplicate_names.index)
print(f"\nrows exposed to a duplicate name (using raw-name duplicate list): {exposed_mask_naive.sum()}")

matches after accent-stripping: 2038 / 2214

rows exposed to a duplicate name (using raw-name duplicate list): 47


## Step 12 — Fix a Normalization Inconsistency, Then Disambiguate by Birth Year

The duplicate-name list (`tm_duplicate_names`) was built on raw accented names, but the exposure check above compares against accent-*stripped* FBref names — an inconsistent basis that under-counts real exposure. Rebuilding the duplicate list on a consistent stripped basis changes the exposed count from 47 to 85 — a meaningful difference. Every exposed match then gets confirmed against birth year rather than trusted at face value.

In [63]:
df_players_tm['name_stripped'] = df_players_tm['name'].apply(strip_accents)
df_players_tm['birth_year'] = pd.to_datetime(df_players_tm['date_of_birth'], errors='coerce').dt.year

tm_name_counts_stripped = df_players_tm['name_stripped'].value_counts()
tm_duplicate_names_stripped = tm_name_counts_stripped[tm_name_counts_stripped > 1]
print(f"duplicate names (stripped basis): {len(tm_duplicate_names_stripped)}  (was {len(tm_duplicate_names)} on raw basis)")

df_players_tm['birth_year'] = pd.to_datetime(df_players_tm['date_of_birth'], errors='coerce').dt.year
tm_lookup_by_name_year = df_players_tm.set_index(['name_stripped', 'birth_year'])['player_id'].to_dict()

def resolve_by_birth_year(row):
    return tm_lookup_by_name_year.get((strip_accents(row['Player']), row['Born']), np.nan)

exposed_mask = matched_mask & df_2425_filtered['Player'].apply(strip_accents).isin(tm_duplicate_names_stripped.index)
exposed_rows = df_2425_filtered[exposed_mask].copy()
exposed_rows['tm_id_by_birthyear'] = exposed_rows.apply(resolve_by_birth_year, axis=1)

agree = (exposed_rows['tm_id_by_birthyear'] == exposed_rows['tm_player_id']).sum()
disagree = (exposed_rows['tm_id_by_birthyear'].notna() & (exposed_rows['tm_id_by_birthyear'] != exposed_rows['tm_player_id'])).sum()
no_match = exposed_rows['tm_id_by_birthyear'].isna().sum()
print(f"\nrows exposed to a duplicate name (corrected): {len(exposed_rows)}")
print(f"agree: {agree} | disagree: {disagree} | no birth-year match: {no_match}")

to_correct = exposed_rows[exposed_rows['tm_id_by_birthyear'].notna() & (exposed_rows['tm_id_by_birthyear'] != exposed_rows['tm_player_id'])]
df_2425_filtered.loc[to_correct.index, 'tm_player_id'] = to_correct['tm_id_by_birthyear']
df_2425_filtered.loc[to_correct.index, 'match_method'] = 'exact_birthyear_corrected'
print(f"corrected {len(to_correct)} misassigned matches")

no_birthyear_rows = exposed_rows[exposed_rows['tm_id_by_birthyear'].isna()]
df_2425_filtered.loc[no_birthyear_rows.index, 'match_method'] = 'exact_duplicate_unverified'
print(f"flagged {len(no_birthyear_rows)} as unverified")

duplicate names (stripped basis): 938  (was 894 on raw basis)

rows exposed to a duplicate name (corrected): 85
agree: 41 | disagree: 41 | no birth-year match: 3
corrected 41 misassigned matches
flagged 3 as unverified


## Step 13 — Investigate the Unverified Duplicate-Name Cases

3 matches have a duplicated name with no birth-year match among the candidates. Look at each one's actual Transfermarkt candidates before deciding whether they're fixable or genuinely unmatchable.

In [64]:
unverified = df_2425_filtered[df_2425_filtered['match_method'] == 'exact_duplicate_unverified']
print(unverified[['Player', 'Born', 'Squad', 'Comp', 'tm_player_id']])

for _, row in unverified.iterrows():
    candidates = df_players_tm[df_players_tm['name_stripped'] == strip_accents(row['Player'])]
    print(f"\n--- {row['Player']} (FBref Born={row['Born']}) ---")
    print(candidates[['player_id', 'name', 'date_of_birth', 'current_club_name', 'market_value_in_eur']])

        Player    Born       Squad        Comp  tm_player_id
726      Diego  2000.0     Leganés  es La Liga        4248.0
777    Douglas  1997.0      Nantes  fr Ligue 1       58426.0
1143  Henrique  1994.0  Valladolid  es La Liga       33572.0

--- Diego (FBref Born=2000.0) ---
       player_id   name        date_of_birth       current_club_name  market_value_in_eur
319         4248  Diego  1985-02-28 00:00:00  Fenerbahçe Spor Kulübü             300000.0
2220       26229  Diego  1979-05-11 00:00:00      Vitória Setúbal FC              50000.0
12348     183646  Diego  1988-05-15 00:00:00                  FK Ufa             250000.0
22687     376155  Diego  1995-09-21 00:00:00     Rakuten Vissel Kobe             450000.0

--- Douglas (FBref Born=1997.0) ---
       player_id     name        date_of_birth           current_club_name  market_value_in_eur
5215       58426  Douglas  1990-08-06 00:00:00   Beşiktaş Jimnastik Kulübü              25000.0
5827       64590  Douglas  1988-01-12 00:0

## Step 14 — Fuzzy Match Remaining Unmatched Players

Applies fuzzy string matching (`thefuzz`, score ≥ 85) to whatever's left after exact and accent-normalized matching.

In [65]:
still_unmatched = df_2425_filtered[df_2425_filtered['tm_player_id'].isna()].copy()
print(f"attempting fuzzy match on {len(still_unmatched)} players")

tm_names_list = df_players_tm['name_stripped'].tolist()

def fuzzy_match_name(name, choices, threshold=85):
    result = process.extractOne(strip_accents(name), choices, scorer=fuzz.token_sort_ratio)
    return (result[0], result[1]) if result and result[1] >= threshold else (None, None)

fuzzy_results = still_unmatched['Player'].apply(lambda n: fuzzy_match_name(n, tm_names_list))
still_unmatched['fuzzy_name'] = fuzzy_results.apply(lambda x: x[0])
still_unmatched['fuzzy_score'] = fuzzy_results.apply(lambda x: x[1])

print(f"fuzzy matches found: {still_unmatched['fuzzy_name'].notna().sum()} / {len(still_unmatched)}")
print(still_unmatched['fuzzy_score'].describe())

attempting fuzzy match on 176 players
fuzzy matches found: 80 / 176
count     80.000000
mean      93.387500
std        4.992383
min       85.000000
25%       90.000000
50%       93.000000
75%       97.000000
max      100.000000
Name: fuzzy_score, dtype: float64


## Step 15 — Sanity-Check Fuzzy Matches via Birth Year, Then Apply

A fuzzy score ≥85 confirms name similarity, not identity. Cross-check each fuzzy match's birth year, then apply all verified matches — including nulling out Diego/Douglas/Henrique per Step 13's finding that none of their duplicate-name candidates are a genuine match.

In [66]:
def verify_fuzzy_birthyear(row):
    candidates = df_players_tm[df_players_tm['name_stripped'] == row['fuzzy_name']]
    if len(candidates) == 0:
        return np.nan, False
    if len(candidates) == 1:
        return candidates.iloc[0]['player_id'], True
    exact_year = candidates[candidates['birth_year'] == row['Born']]
    return (exact_year.iloc[0]['player_id'], True) if len(exact_year) == 1 else (np.nan, False)

fuzzy_matched = still_unmatched[still_unmatched['fuzzy_name'].notna()].copy()
verify_results = fuzzy_matched.apply(verify_fuzzy_birthyear, axis=1, result_type='expand')
fuzzy_matched['verified_tm_id'] = verify_results[0]
fuzzy_matched['verified'] = verify_results[1]

print(f"fuzzy matches verified: {fuzzy_matched['verified'].sum()} / {len(fuzzy_matched)}")
print(fuzzy_matched[~fuzzy_matched['verified']][['Player', 'fuzzy_name', 'fuzzy_score', 'Born']])

# From Step 13: Diego, Douglas, and Henrique had no valid TM candidate —
# null these out rather than keep the unverifiable first-match guess
wrong_mask = df_2425_filtered['match_method'] == 'exact_duplicate_unverified'
df_2425_filtered.loc[wrong_mask, 'tm_player_id'] = np.nan
df_2425_filtered.loc[wrong_mask, 'match_method'] = 'unmatched_needs_manual_review'

verified_fuzzy = fuzzy_matched[fuzzy_matched['verified']]
df_2425_filtered.loc[verified_fuzzy.index, 'tm_player_id'] = verified_fuzzy['verified_tm_id']
df_2425_filtered.loc[verified_fuzzy.index, 'match_method'] = 'fuzzy_verified'

failed_fuzzy = fuzzy_matched[~fuzzy_matched['verified']]
df_2425_filtered.loc[failed_fuzzy.index, 'match_method'] = 'unmatched_needs_manual_review'

no_fuzzy = still_unmatched[still_unmatched['fuzzy_name'].isna()]
df_2425_filtered.loc[no_fuzzy.index, 'match_method'] = df_2425_filtered.loc[no_fuzzy.index, 'match_method'].fillna('unmatched_no_fuzzy_candidate')

print(f"\n=== final match_method breakdown ===")
print(df_2425_filtered['match_method'].value_counts(dropna=False))
total_matched = df_2425_filtered['tm_player_id'].notna().sum()
print(f"\ntotal matched: {total_matched} / {len(df_2425_filtered)} ({total_matched/len(df_2425_filtered):.1%})")

fuzzy matches verified: 79 / 80
    Player fuzzy_name  fuzzy_score    Born
33  Adrián    Adriano         92.0  1987.0

=== final match_method breakdown ===
match_method
exact                            1881
exact_accent_stripped             113
unmatched_no_fuzzy_candidate       96
fuzzy_verified                     79
exact_birthyear_corrected          41
unmatched_needs_manual_review       4
Name: count, dtype: int64

total matched: 2114 / 2214 (95.5%)


## Step 16 — Join Transfermarkt Market Valuations

Joins `player_valuations.csv` on `tm_player_id`, taking the most recent valuation within the 2024-25 window. This becomes the target variable.

In [67]:
df_valuations = pd.read_csv(RAW_TMARKT / "player_valuations.csv")
df_valuations['date'] = pd.to_datetime(df_valuations['date'], errors='coerce')

val_window = df_valuations[(df_valuations['date'] >= '2024-01-01') & (df_valuations['date'] <= '2025-12-31')].copy()
val_latest = val_window.sort_values('date').groupby('player_id').tail(1)

df_merged = df_2425_filtered.merge(
    val_latest[['player_id', 'date', 'market_value_in_eur']],
    left_on='tm_player_id', right_on='player_id', how='left'
)

has_valuation = df_merged['market_value_in_eur'].notna()
print(f"rows with a market value: {has_valuation.sum()} / {len(df_merged)} ({has_valuation.mean():.1%})")
print(df_merged['market_value_in_eur'].describe())

rows with a market value: 2061 / 2214 (93.1%)
count    2.061000e+03
mean     1.383810e+07
std      1.966102e+07
min      5.000000e+04
25%      2.500000e+06
50%      6.500000e+06
75%      1.800000e+07
max      2.000000e+08
Name: market_value_in_eur, dtype: float64


## Step 17 — Filter to Rows with a Valid Target Variable

Drops rows with no market value — they can't be used for supervised training.

In [68]:
df_final = df_merged[df_merged['market_value_in_eur'].notna()].copy()
print(f"after target filter: {len(df_final)}")
print(df_final['position_group'].value_counts(normalize=True))

after target filter: 2061
position_group
DEF    0.377487
MID    0.309073
FWD    0.236293
GK     0.077147
Name: proportion, dtype: float64


## Step 18 — Player Profile Features from Transfermarkt

Pulls date of birth (precise age), nationality, contract expiration, height, and foot from `players.csv`. Uses `age_precise` — not FBref's own `Age` — consistently from here on, since the two use different reference dates.

In [69]:
profile_cols = ['player_id', 'date_of_birth', 'country_of_citizenship', 'contract_expiration_date', 'height_in_cm', 'foot']
df_final = df_final.merge(df_players_tm[profile_cols], left_on='tm_player_id', right_on='player_id', how='left', suffixes=('', '_tm'))

REFERENCE_DATE = pd.Timestamp('2025-01-01')
df_final['date_of_birth'] = pd.to_datetime(df_final['date_of_birth'], errors='coerce')
df_final['contract_expiration_date'] = pd.to_datetime(df_final['contract_expiration_date'], errors='coerce')
df_final['age_precise'] = (REFERENCE_DATE - df_final['date_of_birth']).dt.days / 365.25
df_final['contract_months_remaining'] = (df_final['contract_expiration_date'] - REFERENCE_DATE).dt.days / 30.44

for col in ['date_of_birth', 'country_of_citizenship', 'contract_expiration_date', 'height_in_cm', 'foot', 'age_precise', 'contract_months_remaining']:
    print(f"{col:30s}: {df_final[col].isna().mean():.1%} null")

date_of_birth                 : 0.0% null
country_of_citizenship        : 0.0% null
contract_expiration_date      : 3.6% null
height_in_cm                  : 0.0% null
foot                          : 0.0% null
age_precise                   : 0.0% null
contract_months_remaining     : 3.6% null


## Step 19 — Feature Engineering

Log-transforms the target (heavily right-skewed), adds age-squared for the non-linear value curve, and buckets career stage.

In [70]:
df_final['log_market_value'] = np.log1p(df_final['market_value_in_eur'])
df_final['age_squared'] = df_final['age_precise'] ** 2

def career_stage(age):
    if age < 22: return 'Emerging'
    elif age < 26: return 'Developing'
    elif age < 30: return 'Peak'
    else: return 'Declining'

df_final['career_stage'] = df_final['age_precise'].apply(career_stage)
print(df_final['log_market_value'].describe())
print(f"\n{df_final['career_stage'].value_counts()}")

count    2061.000000
mean       15.662962
std         1.323931
min        10.819798
25%        14.731802
50%        15.687313
75%        16.705882
max        19.113828
Name: log_market_value, dtype: float64

career_stage
Developing    705
Peak          627
Declining     445
Emerging      284
Name: count, dtype: int64


## Step 20 — League Encoding (One-Hot, Not Ordinal)

Deliberately not pre-ranking leagues by assumed quality — that would bake a researcher assumption into the exact question RQ2 tests. One-hot encoding lets the model discover each league's real association with market value.

In [71]:
league_dummies = pd.get_dummies(df_final['Comp'], prefix='league')
df_final = pd.concat([df_final, league_dummies], axis=1)
print(league_dummies.sum())

league_de Bundesliga         384
league_eng Premier League    423
league_es La Liga            421
league_fr Ligue 1            374
league_it Serie A            459
dtype: int64


## Step 21 — Nationality Grouping: Explore the Data First

Before deciding a grouping scheme, check how many nationalities exist and whether any rows are missing `Nation` entirely — a blank field can't be grouped by confederation no matter what scheme is chosen.

In [72]:
print(f"unique nationalities: {df_final['Nation'].nunique()}")
print(f"rows with missing Nation: {df_final['Nation'].isna().sum()}")
print(f"\ntop 20 nationalities by count:\n{df_final['Nation'].value_counts().head(20)}")

unique nationalities: 102
rows with missing Nation: 2

top 20 nationalities by count:
Nation
es ESP     278
fr FRA     236
de GER     177
it ITA     158
eng ENG    153
br BRA      65
ar ARG      58
nl NED      53
pt POR      51
be BEL      42
dk DEN      37
ci CIV      34
ma MAR      33
at AUT      32
ch SUI      31
se SWE      30
hr CRO      28
sn SEN      27
ng NGA      26
rs SRB      25
Name: count, dtype: int64


## Step 22 — Map Nationality to Confederation

102 nationalities is too sparse to one-hot encode individually — group by confederation instead, since ranking nations by assumed "quality" would repeat the same circularity problem the league encoding avoided. Map each 3-letter code and check what's left unmapped before deciding how to handle it.

In [73]:
df_final['nation_code'] = df_final['Nation'].str.split().str[1]

confederation_map = {
    'ESP': 'UEFA', 'FRA': 'UEFA', 'GER': 'UEFA', 'ITA': 'UEFA', 'ENG': 'UEFA', 'NED': 'UEFA', 'POR': 'UEFA',
    'BEL': 'UEFA', 'DEN': 'UEFA', 'AUT': 'UEFA', 'SUI': 'UEFA', 'SWE': 'UEFA', 'CRO': 'UEFA', 'SRB': 'UEFA',
    'POL': 'UEFA', 'UKR': 'UEFA', 'TUR': 'UEFA', 'GRE': 'UEFA', 'SCO': 'UEFA', 'WAL': 'UEFA', 'IRL': 'UEFA',
    'NIR': 'UEFA', 'NOR': 'UEFA', 'FIN': 'UEFA', 'ISL': 'UEFA', 'SVK': 'UEFA', 'SVN': 'UEFA', 'BIH': 'UEFA',
    'ROU': 'UEFA', 'HUN': 'UEFA', 'BUL': 'UEFA', 'ALB': 'UEFA', 'MKD': 'UEFA', 'MNE': 'UEFA', 'KVX': 'UEFA',
    'GEO': 'UEFA', 'ARM': 'UEFA', 'ISR': 'UEFA', 'RUS': 'UEFA', 'LUX': 'UEFA',
    'BRA': 'CONMEBOL', 'ARG': 'CONMEBOL', 'URU': 'CONMEBOL', 'COL': 'CONMEBOL', 'CHI': 'CONMEBOL',
    'PAR': 'CONMEBOL', 'ECU': 'CONMEBOL', 'PER': 'CONMEBOL', 'VEN': 'CONMEBOL', 'BOL': 'CONMEBOL',
    'CIV': 'CAF', 'MAR': 'CAF', 'SEN': 'CAF', 'NGA': 'CAF', 'GHA': 'CAF', 'CMR': 'CAF', 'ALG': 'CAF',
    'TUN': 'CAF', 'MLI': 'CAF', 'GUI': 'CAF', 'RSA': 'CAF', 'EGY': 'CAF', 'ZAM': 'CAF', 'BFA': 'CAF',
    'GAM': 'CAF', 'COD': 'CAF', 'CGO': 'CAF', 'ANG': 'CAF', 'CPV': 'CAF', 'GAB': 'CAF', 'CTA': 'CAF',
    'GNB': 'CAF', 'MOZ': 'CAF', 'ZIM': 'CAF', 'BEN': 'CAF', 'TOG': 'CAF', 'MAD': 'CAF', 'COM': 'CAF',
    'USA': 'CONCACAF', 'MEX': 'CONCACAF', 'CAN': 'CONCACAF', 'CRC': 'CONCACAF', 'JAM': 'CONCACAF',
    'HAI': 'CONCACAF', 'HON': 'CONCACAF', 'PAN': 'CONCACAF', 'CUW': 'CONCACAF', 'TRI': 'CONCACAF',
    'JPN': 'AFC', 'KOR': 'AFC', 'AUS': 'AFC', 'IRN': 'AFC', 'KSA': 'AFC', 'QAT': 'AFC', 'UZB': 'AFC',
    'CHN': 'AFC', 'IRQ': 'AFC', 'SYR': 'AFC',
    'NZL': 'OFC',
}

df_final['confederation'] = df_final['nation_code'].map(confederation_map)
unmapped = df_final[df_final['confederation'].isna()]
print(f"unmapped nation codes: {unmapped['nation_code'].nunique()}")
print(unmapped['nation_code'].value_counts())

unmapped nation codes: 17
nation_code
CZE    12
EQG     3
IDN     2
SLE     2
SUR     2
GUF     2
MAS     1
LTU     1
PHI     1
CYP     1
GLP     1
EST     1
LBY     1
BDI     1
KEN     1
DOM     1
MLT     1
Name: count, dtype: int64


## Step 23 — Patch the Missing Confederations

17 country codes weren't in the initial map — real countries just missing from the dictionary, not data errors. Patch them and check what's still unresolved.

In [74]:
patch_map = {
    'CZE': 'UEFA', 'LTU': 'UEFA', 'CYP': 'UEFA', 'EST': 'UEFA', 'MLT': 'UEFA',
    'EQG': 'CAF', 'SLE': 'CAF', 'LBY': 'CAF', 'BDI': 'CAF', 'KEN': 'CAF',
    'SUR': 'CONMEBOL', 'GUF': 'CONMEBOL',
    'DOM': 'CONCACAF', 'GLP': 'CONCACAF',
    'IDN': 'AFC', 'MAS': 'AFC', 'PHI': 'AFC',
}
df_final['confederation'] = df_final['confederation'].fillna(df_final['nation_code'].map(patch_map))

print(f"unmapped after patch: {df_final['confederation'].isna().sum()}")
print(df_final[df_final['confederation'].isna()][['Player', 'Nation', 'nation_code', 'Squad', 'Comp']])

unmapped after patch: 2
              Player Nation nation_code       Squad           Comp
939   Atakan Karazor    NaN         NaN   Stuttgart  de Bundesliga
1095       Fer López    NaN         NaN  Celta Vigo     es La Liga


## Step 24 — Resolve the Remaining 2 Players Manually

Both remaining unmapped rows have a blank `Nation` field in the source data entirely — not a missing code, but no code at all. Confirmed by manual lookup: Atakan Karazor (Turkish), Fer López (Spanish).

In [75]:
manual_nationality_fix = {
    'Atakan Karazor': {'nation': 'tr TUR', 'code': 'TUR'},
    'Fer López':      {'nation': 'es ESP', 'code': 'ESP'},
}
for player, info in manual_nationality_fix.items():
    mask = df_final['Player'] == player
    df_final.loc[mask, 'Nation'] = info['nation']
    df_final.loc[mask, 'nation_code'] = info['code']
    df_final.loc[mask, 'confederation'] = confederation_map.get(info['code'], patch_map.get(info['code']))

print(f"unmapped after manual fix: {df_final['confederation'].isna().sum()}")
print(df_final['confederation'].value_counts())

unmapped after manual fix: 0
confederation
UEFA        1544
CAF          259
CONMEBOL     176
CONCACAF      41
AFC           38
OFC            3
Name: count, dtype: int64


## Step 25 — One-Hot Encode Confederation

Same rationale as the league encoding — the model discovers each confederation's association with market value rather than being told one upfront.

In [76]:
confederation_dummies = pd.get_dummies(df_final['confederation'], prefix='confed')
df_final = pd.concat([df_final, confederation_dummies], axis=1)
print(confederation_dummies.sum())
print(f"\nduplicate column names? {df_final.columns.duplicated().any()}")

confed_AFC           38
confed_CAF          259
confed_CONCACAF      41
confed_CONMEBOL     176
confed_OFC            3
confed_UEFA        1544
dtype: int64

duplicate column names? False


## Step 26 — Split by Position

Creates `df_fwd`, `df_mid`, `df_def`, `df_gk` using `position_group`.

In [77]:
df_fwd = df_final[df_final['position_group'] == 'FWD'].copy()
df_mid = df_final[df_final['position_group'] == 'MID'].copy()
df_def = df_final[df_final['position_group'] == 'DEF'].copy()
df_gk  = df_final[df_final['position_group'] == 'GK'].copy()

for name, d in [('FWD', df_fwd), ('MID', df_mid), ('DEF', df_def), ('GK', df_gk)]:
    print(f"{name}: {d.shape}")

FWD: (487, 204)
MID: (637, 204)
DEF: (778, 204)
GK: (159, 204)


## Step 27 — Drop Position-Irrelevant Columns

Trims each position dataframe of columns that are ≥90% null within that group — keeper stats for outfield players, and vice versa.

In [78]:
def get_droppable_columns(df_pos, threshold=0.9):
    null_rates = df_pos.isna().mean()
    return null_rates[null_rates >= threshold].index.tolist()

outfield_drop_cols = get_droppable_columns(df_fwd)
gk_drop_cols = get_droppable_columns(df_gk)

df_fwd = df_fwd.drop(columns=outfield_drop_cols)
df_mid = df_mid.drop(columns=outfield_drop_cols)
df_def = df_def.drop(columns=outfield_drop_cols)
df_gk  = df_gk.drop(columns=gk_drop_cols)

for name, d in [('FWD', df_fwd), ('MID', df_mid), ('DEF', df_def), ('GK', df_gk)]:
    print(f"{name}: {d.shape} | remaining >=90% null cols: {len(get_droppable_columns(d))}")

FWD: (487, 167) | remaining >=90% null cols: 0
MID: (637, 167) | remaining >=90% null cols: 0
DEF: (778, 167) | remaining >=90% null cols: 0
GK: (159, 199) | remaining >=90% null cols: 0


## Step 28 — Build the Ablation Dataframe

Restricts to the reduced feature set common with 2025-26, for the feature-ablation experiment.

In [79]:
reduced_feature_cols = [
    '90s', 'Age', 'Born', 'Comp', 'CrdR', 'CrdY', 'Gls', 'Ast', 'MP_playing_time',
    'Min_playing_time', 'Nation', 'PK', 'Pos', 'Starts_playing_time', 'PKatt_shooting',
]
id_and_target_cols = ['Player', 'Squad', 'position_group', 'tm_player_id', 'market_value_in_eur', 'log_market_value', 'age_precise']

ablation_cols = [c for c in reduced_feature_cols + id_and_target_cols if c in df_final.columns]
df_ablation = df_final[ablation_cols].copy()
print(f"df_ablation shape: {df_ablation.shape}")
print(f"duplicate column names? {df_ablation.columns.duplicated().any()}")

df_ablation shape: (2061, 22)
duplicate column names? False


## Step 29 — Save Processed Datasets

Saves the full merged dataset, four position splits, and the ablation dataframe to `data/processed/`.

In [80]:
PROCESSED_DIR = Path("../data/processed/")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_final.to_csv(PROCESSED_DIR / "merged_dataset_2425.csv", index=False)
df_fwd.to_csv(PROCESSED_DIR / "df_fwd.csv", index=False)
df_mid.to_csv(PROCESSED_DIR / "df_mid.csv", index=False)
df_def.to_csv(PROCESSED_DIR / "df_def.csv", index=False)
df_gk.to_csv(PROCESSED_DIR / "df_gk.csv", index=False)
df_ablation.to_csv(PROCESSED_DIR / "df_ablation.csv", index=False)

for f in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"{f.name}  ({f.stat().st_size/1024:.1f} KB)")

for name, d in [('merged_dataset_2425', df_final), ('df_fwd', df_fwd), ('df_mid', df_mid), ('df_def', df_def), ('df_gk', df_gk), ('df_ablation', df_ablation)]:
    print(f"{name:22s}: {d.shape}")

df_ablation.csv  (299.7 KB)
df_def.csv  (563.9 KB)
df_fwd.csv  (353.7 KB)
df_gk.csv  (133.9 KB)
df_mid.csv  (465.1 KB)
merged_dataset_2425.csv  (1581.7 KB)
merged_dataset_2425   : (2061, 204)
df_fwd                : (487, 167)
df_mid                : (637, 167)
df_def                : (778, 167)
df_gk                 : (159, 199)
df_ablation           : (2061, 22)
